# Cost

In [ ]:
#| hide
#| default_exp helpers.cost

In [ ]:
#| exports
import timeit
import random
import pandas as pd
import matplotlib.pyplot as plt

class Cost:
    def __init__(self, repeats=5, seed=0, sizes=(500, 1_000, 2_000)):
        self.repeats = repeats
        self.seed = seed
        self.sizes = sizes
        self.rows = []

    class Counts:
        def __init__(self):
            self.comparisons = 0
            self.predicates = 0
            self.swaps = 0

    class _Item:
        def __init__(self, value, counts):
            self.value, self.counts = value, counts
        def __lt__(self, other):
            self.counts.comparisons += 1
            if type(other) is type(self):
                other = other.value
            return self.value < other

    class _List(list):
        def __init__(self, values, counts):
            super().__init__(values)
            self.counts = counts
        def __setitem__(self, index, value):
            self.counts.swaps += 1
            super().__setitem__(index, value)

    class _Pred:
        def __init__(self, pred, counts):
            self.pred, self.counts = pred, counts
        def __call__(self, *args, **kwargs):
            self.counts.predicates += 1
            return self.pred(*args, **kwargs)
        # def __call__(self, *args):
        #     self.counts.predicates += 1
        #     return self.pred(*args)

    def measure(self, fn, src, *args):
        counts = self.Counts()
        if any(callable(arg) for arg in args):
            data = self._List(src, counts)
            args = [self._Pred(arg, counts) if callable(arg) else arg for arg in args]
        else:
            data = self._List((self._Item(x, counts) for x in src), counts)
        fn(data, *args)
        counts.swaps //= 2
        return counts

    def _ms(self, fn, src, args):
        seq = list(src)
        seconds = timeit.timeit(lambda: fn(seq.copy(), *args), number=self.repeats)
        return seconds / self.repeats * 1_000

    def run(self, fn, src, *args, name=None, shape='', operations=('ms',)):
        if isinstance(operations, str):
            operations = (operations,)
        seq = list(src)
        comparisons = predicates = swaps = ms = None
        counted = [name for name in ('comparisons', 'predicates', 'swaps') if name in operations]
        if counted:
            cost = self.measure(fn, seq, *args)
            if 'comparisons' in counted:
                comparisons = cost.comparisons
            if 'predicates' in counted:
                predicates = cost.predicates
            if 'swaps' in counted:
                swaps = cost.swaps
        if 'ms' in operations:
            ms = round(self._ms(fn, seq, args), 4)
        row = pd.Series({
            'algo': name or fn.__name__,
            'shape': shape,
            'n': len(seq),
            'comparisons': comparisons,
            'predicates': predicates,
            'swaps': swaps,
            'ms': ms,
        })
        self.rows.append(row)
        return row

    def _algos(self, algos):
        if callable(algos):
            return {algos.__name__: algos}
        if isinstance(algos, dict):
            return algos
        return {fn.__name__: fn for fn in algos}

    def scale(self, algos, args=None, sizes=None, operations=('ms',)):
        if args is None:
            args = lambda data: (0, len(data))
        random.seed(self.seed)
        for n in (self.sizes if sizes is None else sizes):
            src = [random.randrange(10_000) for _ in range(n)]
            ordered = sorted(src)
            shapes = {'random': src, 'sorted': ordered, 'reversed': list(reversed(ordered))}
            for shape, data in shapes.items():
                for name, fn in self._algos(algos).items():
                    self.run(fn, data, *args(data), name=name, shape=shape, operations=operations)
        return self.frame

    @property
    def frame(self):
        return pd.DataFrame(self.rows)

    def plot(self, columns=('ms',), frame=None):
        df = self.frame if frame is None else frame
        if isinstance(columns, str):
            columns = (columns,)
        algos = list(dict.fromkeys(df['algo']))
        color = {algo: f'C{i % 10}' for i, algo in enumerate(algos)}
        linestyle = {'random': '-', 'sorted': '--', 'reversed': ':', '': '-'}
        fig, axes = plt.subplots(1, len(columns), figsize=(4.2 * len(columns), 4), squeeze=False)
        lines = df['n'].nunique() > 1
        for ax, col in zip(axes[0], columns):
            for (algo, shape), group in df.groupby(['algo', 'shape'], dropna=False):
                label = algo if shape == '' else f'{algo}, {shape}'
                if lines:
                    group = group.sort_values('n')
                    ax.plot(group['n'], group[col], marker='o', label=label,
                            color=color[algo], linestyle=linestyle.get(shape, '-'))
                else:
                    ax.bar(label, group[col].iloc[0], color=color[algo])
            ax.set_title(col)
            ax.grid(True, axis='y')
            if lines:
                ax.set_xlabel('n')
                if col in ('comparisons', 'predicates'):
                    ax.set_yscale('log')
                ax.legend(fontsize=7)
        fig.tight_layout()